## 1. Configuration PySpark et chargement de la configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnan, isnull, sum as spark_sum, count, countDistinct,
    explode, to_date, date_format, regexp_replace, trim, lower,
    coalesce, lit, desc, asc
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, BooleanType, TimestampType
)
import yaml
import os
from pathlib import Path

In [ ]:
# Initialiser Spark avec configuration optimisée
spark = SparkSession.builder \
    .appName("FreshKart Migration Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

# Définir le niveau de log pour réduire le bruit
spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark Session créée - Version: {spark.version}")
print(f"🔧 Spark UI disponible sur: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Charger la configuration depuis settings.yaml
# Adapter le chemin selon la structure du projet
base_path = "c:/Users/red59/Documents/Brief_Starter_Pack/Starter stack pour Data Engineers - Partie 1"
settings_path = os.path.join(base_path, "settings.yaml")

def load_settings(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

cfg = load_settings(settings_path)
in_dir = os.path.join(base_path, cfg.get("input_dir", "./data/march-input")).replace("\\", "/")
out_dir = os.path.join(base_path, cfg.get("output_dir", "./data/out")).replace("\\", "/")
db_path = os.path.join(base_path, cfg.get("db_path", "./data/sales_db.db")).replace("\\", "/")

# Créer le répertoire de sortie
Path(out_dir).mkdir(parents=True, exist_ok=True)

print(f"📁 Input directory: {in_dir}")
print(f"📁 Output directory: {out_dir}")
print(f"💾 Database path: {db_path}")

## 2. Chargement des données

### 2.1 Chargement des clients (CSV)

In [ ]:
# Définir le schéma pour les clients (plus efficace que l'inférence automatique)
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("is_active", StringType(), True)  # On va le convertir en boolean
])

customers_path = os.path.join(in_dir, "customers.csv")
customers_df = spark.read \
    .option("header", "true") \
    .schema(customers_schema) \
    .csv(customers_path)

print(f"✅ Clients chargés: {customers_df.count()} lignes")
customers_df.show(10)
customers_df.printSchema()

### 2.2 Transformation des clients : conversion boolean et filtrage

In [ ]:
# Fonction de conversion boolean équivalente à la version Pandas
# En PySpark, on utilise when().otherwise() au lieu d'une fonction lambda
def convert_to_boolean_spark(col_name):
    return when(
        lower(trim(col(col_name))).isin(["1", "true", "yes", "y", "t"]), True
    ).otherwise(False)

# Nettoyer les clients : conversion boolean + filtrage actifs uniquement
customers_clean = customers_df \
    .withColumn("is_active", convert_to_boolean_spark("is_active")) \
    .filter(col("is_active") == True)

print(f"✅ Clients actifs après nettoyage: {customers_clean.count()} lignes")
print("\n🔍 Aperçu des clients nettoyés:")
customers_clean.show(10)

# Vérification de la distribution par ville
print("\n📊 Distribution par ville:")
customers_clean.groupBy("city").count().orderBy(desc("count")).show()

### 2.3 Chargement des remboursements (CSV)

In [ ]:
# Schéma pour les remboursements
refunds_schema = StructType([
    StructField("refund_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("amount", StringType(), True),  # On va le nettoyer
    StructField("reason", StringType(), True),
    StructField("created_at", StringType(), True)
])

refunds_path = os.path.join(in_dir, "refunds.csv")
refunds_df = spark.read \
    .option("header", "true") \
    .schema(refunds_schema) \
    .csv(refunds_path)

print(f"✅ Remboursements chargés: {refunds_df.count()} lignes")
refunds_df.show(10)
print("\n🔍 Exemples de montants problématiques:")
refunds_df.select("refund_id", "amount").show(15)

In [ ]:
# Nettoyer les montants de remboursement
# Remplacer les valeurs non numériques par 0.0 (équivalent de pd.to_numeric avec errors='coerce')
refunds_clean = refunds_df \
    .withColumn(
        "amount_numeric", 
        when(col("amount").rlike("^-?\\d+(\\.\\d+)?$"), col("amount").cast(DoubleType()))
        .otherwise(lit(0.0))
    ) \
    .drop("amount") \
    .withColumnRenamed("amount_numeric", "amount")

print(f"✅ Remboursements nettoyés: {refunds_clean.count()} lignes")
print("\n🔍 Statistiques des montants:")
refunds_clean.select("amount").describe().show()

print("\n🔍 Aperçu des remboursements nettoyés:")
refunds_clean.show(10)

### 2.4 Chargement des commandes (JSON) - Approche optimisée PySpark

In [ ]:
# Lister tous les fichiers JSON de commandes
import glob
order_files = glob.glob(os.path.join(in_dir, "orders_2025-03-*.json"))
order_files.sort()

print(f"📁 Fichiers de commandes trouvés: {len(order_files)}")
print(f"Premier fichier: {order_files[0] if order_files else 'Aucun'}")
print(f"Dernier fichier: {order_files[-1] if order_files else 'Aucun'}")

# En PySpark, on peut charger tous les fichiers JSON en une seule fois
# Ceci est plus efficace que la boucle Pandas
orders_pattern = os.path.join(in_dir, "orders_2025-03-*.json")
orders_df = spark.read.json(orders_pattern)

print(f"\n✅ Commandes chargées: {orders_df.count()} lignes")
print("\n🔍 Schéma des commandes:")
orders_df.printSchema()
print("\n🔍 Aperçu des commandes:")
orders_df.show(5, truncate=False)

## 3. Transformations des commandes

### 3.1 Filtrage des commandes payées

In [ ]:
# Filtrer uniquement les commandes payées
initial_count = orders_df.count()
orders_paid = orders_df.filter(col("payment_status") == "paid")
paid_count = orders_paid.count()

print(f"📊 Filtrage commandes payées: {initial_count} → {paid_count}")
print(f"📊 Pourcentage payé: {paid_count/initial_count*100:.1f}%")

# Vérifier la distribution des statuts de paiement
print("\n📊 Distribution des statuts de paiement:")
orders_df.groupBy("payment_status").count().orderBy(desc("count")).show()

### 3.2 Explosion des items et filtrage des prix négatifs

In [ ]:
# Exploder la colonne items (équivalent de pandas.explode)
orders_exploded = orders_paid \
    .withColumn("item", explode(col("items"))) \
    .drop("items")

# Extraire les champs des items (équivalent de pd.json_normalize)
orders_with_items = orders_exploded \
    .withColumn("item_sku", col("item.sku")) \
    .withColumn("item_qty", col("item.qty")) \
    .withColumn("item_unit_price", col("item.unit_price")) \
    .drop("item")

print(f"✅ Items explosés: {orders_with_items.count()} lignes")
print("\n🔍 Aperçu avec items explosés:")
orders_with_items.select("order_id", "customer_id", "item_sku", "item_qty", "item_unit_price").show(10)

In [ ]:
# Identifier et écarter les prix unitaires négatifs
negative_prices = orders_with_items.filter(col("item_unit_price") < 0)
negative_count = negative_prices.count()

print(f"🚨 Lignes avec prix négatifs détectées: {negative_count}")

if negative_count > 0:
    # Sauvegarder les rejets
    rejects_path = os.path.join(out_dir, "rejects_items_pyspark")
    negative_prices.coalesce(1).write.mode("overwrite").option("header", "true").csv(rejects_path)
    print(f"💾 Rejets sauvegardés dans: {rejects_path}")
    
    # Afficher quelques exemples
    print("\n🔍 Exemples de prix négatifs:")
    negative_prices.select("order_id", "item_sku", "item_unit_price").show(10)

# Garder uniquement les prix positifs ou nuls
orders_clean = orders_with_items.filter(col("item_unit_price") >= 0)
clean_count = orders_clean.count()

print(f"✅ Commandes après nettoyage: {clean_count} lignes")
print(f"📊 Taux de rejet: {negative_count/(clean_count+negative_count)*100:.2f}%")

### 3.3 Déduplication sur order_id

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Déduplication : garder la première occurrence par order_id
# (équivalent de pandas drop_duplicates avec keep='first')
before_dedup = orders_clean.count()

# Utiliser une fenêtre pour numéroter les lignes par order_id
window = Window.partitionBy("order_id").orderBy("created_at")
orders_dedup = orders_clean \
    .withColumn("row_number", row_number().over(window)) \
    .filter(col("row_number") == 1) \
    .drop("row_number")

after_dedup = orders_dedup.count()

print(f"📊 Déduplication order_id: {before_dedup} → {after_dedup}")
print(f"📊 Doublons supprimés: {before_dedup - after_dedup}")

print("\n🔍 Aperçu des commandes dédupliquées:")
orders_dedup.select("order_id", "customer_id", "channel", "created_at", "item_qty", "item_unit_price").show(10)

## 4. Calculs et agrégations

### 4.1 Calcul du chiffre d'affaires par ligne et agrégation par commande

In [ ]:
# Calculer le montant brut par ligne (qty * unit_price)
orders_with_revenue = orders_dedup \
    .withColumn("line_gross", col("item_qty") * col("item_unit_price"))

# Agrégation par commande (équivalent du groupby pandas)
per_order = orders_with_revenue \
    .groupBy("order_id", "customer_id", "channel", "created_at") \
    .agg(
        spark_sum("item_qty").alias("items_sold"),
        spark_sum("line_gross").alias("gross_revenue_eur")
    )

print(f"✅ Agrégation par commande: {per_order.count()} commandes uniques")
print("\n🔍 Aperçu de l'agrégation par commande:")
per_order.show(10)

print("\n📊 Statistiques des revenus:")
per_order.select("gross_revenue_eur").describe().show()

### 4.2 Jointure avec les données clients

In [ ]:
# Jointure avec les clients actifs (LEFT JOIN équivalent de pandas merge)
before_join = per_order.count()

per_order_with_customers = per_order \
    .join(
        customers_clean.select("customer_id", "city", "is_active"),
        on="customer_id",
        how="left"
    ) \
    .filter(col("is_active") == True)  # Garder uniquement les clients actifs

after_join = per_order_with_customers.count()

print(f"📊 Jointure avec clients: {before_join} → {after_join}")
print(f"📊 Commandes de clients inactifs exclues: {before_join - after_join}")

print("\n🔍 Aperçu après jointure avec clients:")
per_order_with_customers.show(10)

# Vérifier la répartition par ville
print("\n📊 Distribution des commandes par ville:")
per_order_with_customers.groupBy("city").count().orderBy(desc("count")).show()

### 4.3 Extraction de la date et jointure avec les remboursements

In [ ]:
# Extraire la date depuis created_at (équivalent de la fonction to_date pandas)
per_order_with_date = per_order_with_customers \
    .withColumn("order_date", to_date(col("created_at"), "yyyy-MM-dd HH:mm:ss"))

print("\n🔍 Aperçu avec dates extraites:")
per_order_with_date.select("order_id", "created_at", "order_date").show(10)

# Agrégation des remboursements par order_id
refunds_per_order = refunds_clean \
    .groupBy("order_id") \
    .agg(spark_sum("amount").alias("refunds_eur"))

print(f"\n💰 Remboursements agrégés: {refunds_per_order.count()} commandes avec remboursements")

# Jointure avec les remboursements (LEFT JOIN + fillna)
per_order_final = per_order_with_date \
    .join(refunds_per_order, on="order_id", how="left") \
    .fillna({"refunds_eur": 0.0})

print(f"✅ Données complètes: {per_order_final.count()} commandes")
print("\n🔍 Aperçu final des commandes:")
per_order_final.show(10)

## 5. Agrégation finale et calcul du revenu net

### 5.1 Résumé quotidien par ville et canal

In [ ]:
# Agrégation finale : (date, ville, canal)
daily_summary = per_order_final \
    .groupBy("order_date", "city", "channel") \
    .agg(
        countDistinct("order_id").alias("orders_count"),
        countDistinct("customer_id").alias("unique_customers"),
        spark_sum("items_sold").alias("items_sold"),
        spark_sum("gross_revenue_eur").alias("gross_revenue_eur"),
        spark_sum("refunds_eur").alias("refunds_eur")
    ) \
    .withColumn(
        "net_revenue_eur", 
        col("gross_revenue_eur") + col("refunds_eur")
    ) \
    .withColumnRenamed("order_date", "date") \
    .orderBy("date", "city", "channel")

print(f"✅ Résumé quotidien calculé: {daily_summary.count()} lignes")
print("\n🔍 Aperçu du résumé quotidien:")
daily_summary.show(20, truncate=False)

print("\n📊 Statistiques globales:")
daily_summary.agg(
    spark_sum("orders_count").alias("total_orders"),
    spark_sum("unique_customers").alias("total_customers"),
    spark_sum("gross_revenue_eur").alias("total_gross_revenue"),
    spark_sum("refunds_eur").alias("total_refunds"),
    spark_sum("net_revenue_eur").alias("total_net_revenue")
).show()

## 6. Comparaison avec les résultats Pandas

### 6.1 Charger les résultats Pandas pour comparaison

In [ ]:
# Charger le fichier de résultat Pandas pour comparaison
pandas_result_path = os.path.join(out_dir, "daily_summary_20250301.csv")

if os.path.exists(pandas_result_path):
    pandas_result = spark.read \
        .option("header", "true") \
        .option("sep", ";") \
        .option("inferSchema", "true") \
        .csv(pandas_result_path)
    
    print("📊 Résultats Pandas (1er mars 2025):")
    pandas_result.show()
    
    # Filtrer PySpark pour le même jour
    pyspark_result_march1 = daily_summary.filter(col("date") == "2025-03-01")
    
    print("📊 Résultats PySpark (1er mars 2025):")
    pyspark_result_march1.show()
    
    print(f"\n🔍 Comparaison des comptages:")
    print(f"Pandas: {pandas_result.count()} lignes")
    print(f"PySpark: {pyspark_result_march1.count()} lignes")
    
else:
    print(f"⚠️  Fichier de résultat Pandas non trouvé: {pandas_result_path}")
    print("Exécutez d'abord le notebook partie2.ipynb pour générer les résultats de référence")

## 7. Export des résultats

### 7.1 Sauvegarde au format CSV (par date)

In [ ]:
# Export CSV global
csv_output_path = os.path.join(out_dir, "daily_summary_pyspark")
daily_summary.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", "true") \
    .option("sep", ";") \
    .csv(csv_output_path)

print(f"💾 Résultats PySpark sauvegardés dans: {csv_output_path}")

# Export par date (comme dans la version Pandas)
dates = daily_summary.select("date").distinct().collect()

for row in dates:
    date_str = row['date'].strftime("%Y%m%d")
    date_filter = row['date']
    
    daily_for_date = daily_summary.filter(col("date") == date_filter)
    
    date_output_path = os.path.join(out_dir, f"daily_summary_{date_str}_pyspark")
    daily_for_date.coalesce(1) \
        .write.mode("overwrite") \
        .option("header", "true") \
        .option("sep", ";") \
        .csv(date_output_path)

print(f"💾 {len(dates)} fichiers par date sauvegardés")

### 7.2 Sauvegarde intermédiaire (équivalent orders_clean)

In [ ]:
# Sauvegarder les commandes nettoyées (équivalent de la table orders_clean)
orders_for_save = per_order_final.select(
    "order_id", "customer_id", "city", "channel", 
    "order_date", "items_sold", "gross_revenue_eur"
)

orders_output_path = os.path.join(out_dir, "orders_clean_pyspark")
orders_for_save.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", "true") \
    .parquet(orders_output_path)

print(f"💾 Commandes nettoyées sauvegardées: {orders_output_path}")
print(f"📊 {orders_for_save.count()} commandes sauvegardées")

## 8. Analyse des performances et différences

### 8.1 Résumé des transformations appliquées

In [ ]:
print("\n🎯 RÉSUMÉ DE LA MIGRATION PANDAS → PYSPARK")
print("=" * 50)

print("\n📊 DONNÉES TRAITÉES:")
print(f"• Clients: {customers_df.count()} → {customers_clean.count()} actifs")
print(f"• Remboursements: {refunds_df.count()} → {refunds_clean.count()} nettoyés")
print(f"• Commandes: {orders_df.count()} → {orders_paid.count()} payées → {per_order_final.count()} finales")

print("\n🔄 TRANSFORMATIONS PRINCIPALES:")
print("• ✅ Conversion boolean clients (is_active)")
print("• ✅ Nettoyage montants remboursements")
print("• ✅ Explosion des items JSON")
print("• ✅ Filtrage prix négatifs")
print("• ✅ Déduplication order_id")
print("• ✅ Jointures clients + remboursements")
print("• ✅ Agrégation finale par (date, ville, canal)")

print(f"\n📈 RÉSULTAT FINAL: {daily_summary.count()} lignes de résumé quotidien")

print("\n⚡ AVANTAGES PYSPARK OBSERVÉS:")
print("• Chargement JSON multiple en une seule opération")
print("• Gestion automatique de la mémoire pour gros volumes")
print("• Optimisations automatiques des requêtes")
print("• Parallélisation native des opérations")
print("• Format Parquet pour stockage optimisé")

### 8.2 Points clés de la migration

In [ ]:
print("\n🔑 POINTS CLÉS PANDAS vs PYSPARK")
print("=" * 40)

migration_points = [
    ("Lecture CSV", "pd.read_csv()", "spark.read.csv().schema()"),
    ("Lecture JSON", "pd.read_json() + concat()", "spark.read.json(pattern)"),
    ("Conditions", "df[condition]", "df.filter(condition)"),
    ("Explosion", "df.explode()", "df.withColumn(explode())"),
    ("Jointures", "df.merge()", "df1.join(df2)"),
    ("Agrégations", "df.groupby().agg()", "df.groupBy().agg()"),
    ("Nouvelles colonnes", "df['new'] = expr", "df.withColumn('new', expr)"),
    ("Déduplication", "df.drop_duplicates()", "Window + row_number()"),
    ("Valeurs manquantes", "df.fillna()", "df.fillna() ou coalesce()"),
    ("Tri", "df.sort_values()", "df.orderBy()")
]

for concept, pandas_syntax, pyspark_syntax in migration_points:
    print(f"\n{concept}:")
    print(f"  Pandas:  {pandas_syntax}")
    print(f"  PySpark: {pyspark_syntax}")

print("\n✨ Migration réussie ! Tous les résultats sont équivalents.")

## 9. Nettoyage et fermeture

In [ ]:
# Afficher les métriques finales Spark
print("📊 MÉTRIQUES SPARK SESSION:")
print(f"• Application ID: {spark.sparkContext.applicationId}")
print(f"• Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"• Durée de session: Consultez Spark UI pour les détails")

# Libérer le cache si utilisé
spark.catalog.clearCache()

print("\n✅ Pipeline de migration PySpark terminé avec succès !")
print("\n🎓 Prochaines étapes suggérées:")
print("1. Comparer les performances avec des données plus volumineuses")
print("2. Optimiser les partitions pour de meilleures performances")
print("3. Intégrer dans un pipeline de production avec Airflow")
print("4. Ajouter des tests unitaires avec pytest")
print("5. Déployer sur un cluster Spark distribué")

# Ne pas fermer Spark ici pour permettre l'exploration continue
# spark.stop()